In [ ]:
import pandas as pd
import numpy as np
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import scipy.sparse as sp
import warnings
warnings.filterwarnings('ignore')

print("1. Loading Raw Dataset...")
df = pd.read_csv('fake reviews dataset.csv')
df.dropna(inplace=True)

print("2. Extracting Numerical Features...")
df['char_count'] = df['text_'].apply(len)
df['word_count'] = df['text_'].apply(lambda x: len(str(x).split()))

def punct_count(text):
    count = sum([1 for char in str(text) if char in string.punctuation])
    return count / len(str(text)) if len(str(text)) > 0 else 0
df['punct_density'] = df['text_'].apply(punct_count)

def cap_ratio(text):
    caps = sum([1 for char in str(text) if char.isupper()])
    return caps / len(str(text)) if len(str(text)) > 0 else 0
df['cap_ratio'] = df['text_'].apply(cap_ratio)

# Encode Label: Fake/CG = 0, Genuine/OR = 1
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

print("3. Splitting Data (80/20) to freeze the alignment...")
# UPDATED: test_size=0.20 for 80-20 split!
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label_encoded'])

print("4. Saving Base Data Files (For future RoBERTa/Fusion use)...")
train_df[['text_', 'label_encoded']].rename(columns={'label_encoded': 'label'}).to_csv('train_text_data.csv', index=False)
test_df[['text_', 'label_encoded']].rename(columns={'label_encoded': 'label'}).to_csv('test_text_data.csv', index=False)

num_features = ['char_count', 'word_count', 'punct_density', 'cap_ratio', 'rating', 'label_encoded']
train_df[num_features].rename(columns={'label_encoded': 'label'}).to_csv('train_num_data.csv', index=False)
test_df[num_features].rename(columns={'label_encoded': 'label'}).to_csv('test_num_data.csv', index=False)

print("\n--- BUILDING THE ML MODEL (Branch B) ---")
# 5. Extract features for ML
X_train_num = train_df[['char_count', 'word_count', 'punct_density', 'cap_ratio', 'rating']].values
X_test_num = test_df[['char_count', 'word_count', 'punct_density', 'cap_ratio', 'rating']].values
y_train = train_df['label_encoded'].values
y_test = test_df['label_encoded'].values

# Scale the numerical features
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

# Vectorize the text using TF-IDF
print("Vectorizing Text with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_df['text_'])
X_test_tfidf = tfidf.transform(test_df['text_'])

# STRICTLY COMBINE TF-IDF AND NUMERICAL DATA
print("Combining TF-IDF with Numerical Data...")
X_train_combined = sp.hstack((X_train_tfidf, X_train_num_scaled))
X_test_combined = sp.hstack((X_test_tfidf, X_test_num_scaled))

# Train the ML Model
print("Training ML Model (Logistic Regression)...")
ml_model = LogisticRegression(max_iter=1000, random_state=42)
ml_model.fit(X_train_combined, y_train)

# Evaluate
ml_preds = ml_model.predict(X_test_combined)
acc = accuracy_score(y_test, ml_preds)
print(f"\n✅ ML Model Standalone Accuracy: {acc * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, ml_preds, target_names=['Fake', 'Genuine']))

print("\n5. Saving the ML Model's Probabilities...")
train_ml_prob = ml_model.predict_proba(X_train_combined)[:, 1]
test_ml_prob = ml_model.predict_proba(X_test_combined)[:, 1]

# UPDATED: Renamed column to lr_prob and files to lr_probabilities
pd.DataFrame({'lr_prob': train_ml_prob}).to_csv('train_lr_probabilities.csv', index=False)
pd.DataFrame({'lr_prob': test_ml_prob}).to_csv('test_lr_probabilities.csv', index=False)

print("✅ All Branch B probabilities successfully saved! Ready for next step.")

1. Loading Raw Dataset...
2. Extracting Numerical Features...
3. Splitting Data (80/20) to freeze the alignment...
4. Saving Base Data Files (For future RoBERTa/Fusion use)...

--- BUILDING THE ML MODEL (Branch B) ---
Vectorizing Text with TF-IDF...
Combining TF-IDF with Numerical Data...
Training ML Model (Logistic Regression)...

✅ ML Model Standalone Accuracy: 87.14%

Classification Report:
               precision    recall  f1-score   support

        Fake       0.88      0.87      0.87      4044
     Genuine       0.87      0.88      0.87      4043

    accuracy                           0.87      8087
   macro avg       0.87      0.87      0.87      8087
weighted avg       0.87      0.87      0.87      8087


5. Saving the ML Model's Probabilities...
✅ All Branch B probabilities successfully saved! Ready for next step.


In [ ]:
import joblib

# Save the Logistic Regression model
joblib.dump(ml_model, 'ml_logistic_regression_model.joblib')
print("✅ Logistic Regression model saved successfully as 'ml_logistic_regression_model.joblib'!")

✅ Logistic Regression model saved successfully as 'ml_logistic_regression_model.joblib'!
